# Run-wise preprocessing with calibration-only ICA (sessions T and E, from the original GDF files)

Preprocessing notebook for the **within-subject cross-session** experiment on BCI Competition IV-2a. It starts from the original **GDF files** (`A0xT.gdf`, `A0xE.gdf`), which contain every paradigm event, and from the official labels in `true_labels/`.

Each run is filtered separately, and an ICA decomposition is estimated **only on the initial EOG-calibration runs** of the session, then frozen and applied to all runs.

It is **not certified as strictly causal** and it is **not shown to be equivalent to a streaming implementation**. The minimum-phase FIR filter is applied offline with MNE's default padding, which does not by itself demonstrate causality at run edges

**Dataset facts used** (official description `desc_2a.pdf`, and checks on the files)
- Each session starts with about 5 minutes of EOG calibration: 2 min eyes open, 1 min eyes closed, 1 min eye movements (exception: in `A04T` this block is shorter and contains only the eye-movement condition). Then 6 runs of 48 trials (12 per class).
- A trial starts at t = 0 s (event 768); the **cue appears at t = 2 s** (events 769–772 with the class in session `T`; event 783, "unknown cue", in session `E`). In every checked file the cue − trial-start distance is exactly **500 samples (2.000 s)**; this is asserted below.
- Session `E` labels are not stored in the GDF: they are read from `true_labels/A0xE.mat` (`classlabel`, in trial order).
- The GDF files contain 100 overflow samples between runs (−1600 µV in the EEG channels, −16000 µV in the EOG channels). They are removed and never reach the filter.
- The EOG channels are provided for artifact removal and must not be used for classification.

**Procedure (per session)**
1. Split the GDF into runs (events 32766) and drop the overflow samples.
2. Identify the calibration runs from the events 276 / 277 / 1072 and check that they precede every task run.
3. Filter each run separately, 1–40 Hz, with a minimum-phase FIR (`phase='minimum'`, default padding).
4. Fit the ICA (20 components, extended infomax, `random_state=42`) **only on the calibration runs**; pick ocular components by automatic correlation with the 3 EOG channels; freeze the decomposition and apply it to every run.
5. Session `E`: by default the ICA is fitted on the calibration runs of `E` (unlabeled, recorded before the trials). The alternative `'session_T_calibration'` applies the frozen decomposition of `T` and is an **exploratory sensitivity analysis**; it must not be chosen by looking at test performance. Each setting writes to its own sub-directory.
6. Save the cleaned `.fif` (class annotations at the cue), a trial table (`*_trials.csv`) and a provenance file (`*_provenance.json`) with a fingerprint of everything that determines the outputs.

**ICA quality control (reported, never used to tune parameters or to choose seeds).** four separate blocks are reported and kept apart:
1. Calibration conditions, durations and number of fit samples.
2. A second fit on the same calibration data with another seed; spatial patterns are matched and compared. It says how reproducible the decomposition is, nothing about its relation with the eye signals.
3. For each excluded component, its score with each of the three EOG channels as returned by MNE's `find_bads_eog` (default settings).
4. Retained RMS amplitude of the EEG channels and mean absolute EEG–EOG correlation before/after cleaning, on the calibration runs (data used for the fit) and on the task runs (not used for the fit, unlabeled, no class information).

The ICA fit counter (`n_iter`, `max_iter`) is also stored together with the **MNE version actually used** (recorded at run time). On its own this counter does **not** identify why the algorithm stopped (iteration limit, convergence criterion, or other internal stopping/restart rules) and it does **not** certify the quality of the decomposition; it is stored only as a descriptive record. Flags are descriptive, their thresholds are conventions that were not tuned, and no seed, parameter or setting is chosen from them or from any session-`E` classification metric.

**Limits.** The calibration runs of session `E` belong to the test session (unlabeled, recorded before the trials). The paradigm timing is known. Pipeline choices were made offline. With calibration data only (especially `A04T`, whose calibration block is shorter) the artifact removal must be inspected per participant.

In [ ]:
import os, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linear_sum_assignment
import mne
from mne.preprocessing import ICA, read_ica

mne.set_log_level('WARNING')

# ---------------- configuration  ----------------
DATA_PATH = Path(os.environ.get('BCI_DATA_DIR', '.'))                       # A01T.gdf, A01E.gdf, ... (same folder as the notebook)
LABELS_PATH = DATA_PATH / 'true_labels'                                      # A01E.mat, ... (variable `classlabel`)
SUBJECTS = list(range(1, 10))
SESSIONS = ['T', 'E']                 # T must be processed before E (needed by the 'session_T_calibration' option)
SKIP_EXISTING = True                  # outputs are reused ONLY if their provenance fingerprint matches the current configuration
FS = 250.0
BAND = (1.0, 40.0)
FILTER_PHASE = 'minimum'              # minimum-phase FIR, applied offline run by run with MNE's default padding.
                                      # this alone does not demonstrate causality at run edges nor streaming equivalence.
N_ICA_COMPONENTS = 20
ICA_SEED = 42
ICA_STABILITY_SEED = 43               # second fit used only for the stability check
ICA_SOURCE_E = 'own_calibration'      # 'own_calibration' (default) | 'session_T_calibration' (exploratory sensitivity analysis)
assert ICA_SOURCE_E in ('own_calibration', 'session_T_calibration')

PIPELINE_VERSION = 'runwise_calibration_ica_v3'
BASE_OUTPUT_PATH = Path(os.environ.get('BCI_CLEAN_DIR', 'cleaned_data_1_40_runwise'))
OUTPUT_PATH = BASE_OUTPUT_PATH / ICA_SOURCE_E          # separate directory for each ICA_SOURCE_E setting
CLEAN_SUFFIX = '_runwise_clean_1_40_raw.fif'
GAP = 100                             # overflow samples inserted between runs in the GDF files
# overflow marker = negative maximum stored value: -1600 uV for the 22 EEG channels (100 uV range), -16000 uV for the 3 EOG (1 mV range)
OVERFLOW_V = np.array([-1599e-6] * 22 + [-15999e-6] * 3)[:, None]
CUE_AFTER_START = 500                 # the cue appears 2.0 s (500 samples) after the trial start (asserted below)
REQUIRED_SPAN_S = (-1.0, 3.5)         # data needed around the cue by the classification notebook: 1 s baseline + 0.5-3.5 s window

# Official channel order: 22 EEG + 3 EOG (the GDF stores generic names such as 'EEG-0')
CH_NAMES = ['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2',
            'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz', 'P2', 'POz',
            'EOG-left', 'EOG-central', 'EOG-right']
EOG_NAMES = CH_NAMES[22:]
CLASS_DESC = {1: 'left', 2: 'right', 3: 'foot', 4: 'tongue'}      # class code (1..4) -> description
CALIB_EVENTS = {276: 'eyes open', 277: 'eyes closed', 1072: 'eye movements'}
STANDARD_CALIBRATION = ['eyes open', 'eyes closed', 'eye movements']
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
print('GDF folder:', DATA_PATH.resolve(), '| outputs:', OUTPUT_PATH.resolve(), '| MNE version in use:', mne.__version__)

GDF folder: C:\Users\ACER\Desktop\split_sessione | outputs: C:\Users\ACER\Desktop\split_sessione\cleaned_data_1_40_runwise\own_calibration | MNE version in use: 1.12.0


## File audit

For each participant and session: presence of the GDF file, of the official labels and of the files already produced.

In [2]:
audit = pd.DataFrame([dict(subject=s, session=ses,
                           gdf=(DATA_PATH / f'A{s:02d}{ses}.gdf').exists(),
                           true_labels=(LABELS_PATH / f'A{s:02d}{ses}.mat').exists(),
                           clean_fif=(OUTPUT_PATH / f'A{s:02d}{ses}{CLEAN_SUFFIX}').exists())
                      for s in SUBJECTS for ses in SESSIONS])
display(audit)
print('cwd:', Path.cwd(), '| GDF files found:', len(list(DATA_PATH.glob('*.gdf'))))

,subject,session,gdf,true_labels,clean_fif
0,1,T,True,True,False
1,1,E,True,True,False
2,2,T,True,True,False
3,2,E,True,True,False
4,3,T,True,True,False
5,3,E,True,True,False
6,4,T,True,True,False
7,4,E,True,True,False
8,5,T,True,True,False
9,5,E,True,True,False


cwd: c:\Users\ACER\Desktop\split_sessione | GDF files found: 18


## Reading a GDF session: channel types, runs, calibration blocks, cue events and labels

- After renaming the generic GDF channel names, **channel types are assigned explicitly** and the notebook asserts 22 EEG + 3 EOG. The mapping of generic names to 10-20 names is positional (documented channel order).
- **Runs** are delimited by events 32766; the 100 overflow samples at the start of each run (except the first) are dropped, after asserting that they are overflow values on all 25 channels and that no overflow sample remains inside a run.
- **Calibration blocks** are the runs without trial events. Each must carry exactly one calibration event (276 eyes open, 277 eyes closed, 1072 eye movements) located at its run start; the calibration runs must be the first runs, must follow the documented sequence (the only allowed deviation is the documented shorter `A04T` block, which is flagged), and must end before the first trial starts.
- **Trials**: cue exactly 500 samples after the trial start; labels of `T` come from the cue events 769–772 (cross-checked with `true_labels/` when present); labels of `E` come only from `true_labels/A0xE.mat`. If missing, the notebook stops: it never invents classes.
- **Epoch containment and finite values**: the data required around every cue (1 s before to 3.5 s after) must lie inside the correct run, and the recording must contain no NaN/inf.

In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def read_gdf_session(path):
    '''Load a GDF file as MNE Raw (25 channels, volts); return the event positions/types as integers.'''
    raw = mne.io.read_raw_gdf(path, preload=True, verbose=False)
    orig = list(raw.ch_names)
    assert len(orig) == 25 and int(raw.info['sfreq']) == int(FS), (len(orig), raw.info['sfreq'])
    assert orig[22:] == EOG_NAMES and all(n.startswith('EEG') for n in orig[:22]), f'unexpected GDF channel layout: {orig}'
    raw.rename_channels(dict(zip(orig, CH_NAMES)))             # generic GDF names -> official 10-20 names (positional)
    # explicit channel types AFTER the renaming, then verify 22 EEG + 3 EOG
    raw.set_channel_types({**{n: 'eeg' for n in CH_NAMES[:22]}, **{n: 'eog' for n in EOG_NAMES}})
    assert raw.get_channel_types() == ['eeg'] * 22 + ['eog'] * 3
    assert len(mne.pick_types(raw.info, eeg=True, eog=False)) == 22
    assert len(mne.pick_types(raw.info, eeg=False, eog=True)) == 3
    assert list(raw.ch_names) == CH_NAMES
    ann = raw.annotations
    pos = np.round(ann.onset * FS).astype(int)
    typ = np.array([int(d) for d in ann.description])
    return raw, pos, typ

def split_runs(raw, pos, typ):
    '''Split the recording into runs (event 32766) and drop the 100 overflow samples at the start of each run.'''
    data = raw.get_data()
    starts = np.sort(pos[typ == 32766])
    assert starts[0] == 0, 'first run must start at sample 0'
    data_start = [0] + [int(s) + GAP for s in starts[1:]]
    data_end = [int(s) for s in starts[1:]] + [data.shape[1]]
    overflow = (data <= OVERFLOW_V).all(axis=0)                  # all 25 channels at their overflow value
    in_run = np.zeros(data.shape[1], dtype=bool)
    for r in range(len(starts)):
        if r > 0:
            assert overflow[starts[r]:starts[r] + GAP].all(), f'run {r}: expected {GAP} overflow samples at its start'
        in_run[data_start[r]:data_end[r]] = True
    assert not overflow[in_run].any(), 'overflow samples found inside a run'
    assert np.isfinite(data[:, in_run]).all(), 'NaN/inf found inside a run'
    return data, starts, data_start, data_end

def identify_calibration(sid, session, pos, typ, starts, data_start, data_end, t_start, task_runs):
    '''Verify that the calibration blocks are identified by the correct events and precede every task run.'''
    n_runs = len(data_start)
    calib_runs = [r for r in range(n_runs) if r not in task_runs]
    assert len(calib_runs) >= 1 and calib_runs == list(range(len(calib_runs))), \
        f'calibration runs must be the first runs, got {calib_runs}'
    cond = {}
    for r in calib_runs:
        here = [int(t) for p, t in zip(pos, typ) if t in CALIB_EVENTS and p == starts[r]]
        assert len(here) == 1, f'run {r}: expected exactly one calibration event (276/277/1072) at its start, found {here}'
        cond[r] = CALIB_EVENTS[here[0]]
    calib_positions = {int(starts[r]) for r in calib_runs}
    assert all(int(p) in calib_positions for p, t in zip(pos, typ) if t in CALIB_EVENTS), \
        'a calibration event lies outside the calibration runs'
    assert data_end[calib_runs[-1]] <= int(t_start.min()), 'calibration does not precede the first task trial'
    conds = [cond[r] for r in calib_runs]
    non_standard = conds != STANDARD_CALIBRATION
    if non_standard:
        # the only documented deviation: A04T has a shorter EOG block with the eye-movement condition only
        assert sid == 4 and session == 'T' and 'eye movements' in conds, f'unexpected calibration structure: {conds}'
    return calib_runs, cond, non_standard

def build_trial_table(sid, session, pos, typ, data_start, data_end):
    '''Trial table in the coordinates of the concatenated (gap-free) recording, with run boundaries.'''
    t_start = np.sort(pos[typ == 768])
    cue_types = [769, 770, 771, 772] if session == 'T' else [783]
    order = np.argsort(pos); cue_mask = np.isin(typ[order], cue_types)
    cue_pos, cue_typ = pos[order][cue_mask], typ[order][cue_mask]
    assert len(t_start) == 288 and len(cue_pos) == 288, (len(t_start), len(cue_pos))
    assert (cue_pos - t_start == CUE_AFTER_START).all(), 'cue is not exactly 2.0 s after the trial start in every trial'

    ds, de = np.array(data_start), np.array(data_end)
    run_of_trial = np.searchsorted(ds, t_start, side='right') - 1
    assert (cue_pos < de[run_of_trial]).all() and (t_start >= ds[run_of_trial]).all(), 'trial not inside its run'
    lengths = de - ds
    offsets = np.concatenate([[0], np.cumsum(lengths)[:-1]])          # start of each run in the concatenated recording
    cue_concat = offsets[run_of_trial] + (cue_pos - ds[run_of_trial])
    start_concat = offsets[run_of_trial] + (t_start - ds[run_of_trial])
    run_start_concat, run_end_concat = offsets[run_of_trial], offsets[run_of_trial] + lengths[run_of_trial]

    # labels: session T from the cue events, session E ONLY from the official label file
    ext_path = LABELS_PATH / f'A{sid:02d}{session}.mat'
    ext = sio.loadmat(ext_path)['classlabel'].ravel().astype(int) if ext_path.exists() else None
    if session == 'T':
        labels, source = cue_typ - 768, 'cue events 769-772'
        if ext is not None:
            assert np.array_equal(ext, labels), 'cue-event labels differ from the true_labels file'
            source += ' (verified against true_labels)'
    else:
        if ext is None:
            raise FileNotFoundError(f'{ext_path} not found: session E labels are not stored in the GDF file.')
        assert len(ext) == 288 and np.isin(ext, [1, 2, 3, 4]).all()
        labels, source = ext, 'true_labels file'

    artifact = np.isin(t_start, pos[typ == 1023]).astype(int)       # expert-marked artifact trials (flag only)
    task_runs = sorted(set(run_of_trial.tolist()))
    run_idx = np.array([task_runs.index(r) for r in run_of_trial])
    trial_in_run = np.concatenate([np.arange((run_idx == k).sum()) for k in range(len(task_runs))])
    table = pd.DataFrame(dict(subject=sid, session=session, trial_in_session=np.arange(288), run=run_idx,
                              trial_in_run=trial_in_run, trial_start_sample=start_concat, cue_sample=cue_concat,
                              run_start_sample=run_start_concat, run_end_sample=run_end_concat,
                              label_code=labels, artifact_flag=artifact, label_source=source))
    # all data required around every cue must lie inside the correct run
    before, after = int(round(REQUIRED_SPAN_S[0] * FS)), int(round(REQUIRED_SPAN_S[1] * FS))
    assert (table['cue_sample'] + before >= table['run_start_sample']).all(), 'baseline would start before its run'
    assert (table['cue_sample'] + after < table['run_end_sample']).all(), 'window would extend past its run'
    return table, task_runs

## ICA fitting, quality control and output compatibility

- `fit_ica` fits the extended-infomax ICA on the calibration runs and selects ocular components by correlation with the EOG channels (MNE `find_bads_eog`, default settings). Fit warnings are recorded.
- `ica_stability` is the **stability-between-seeds** block: it refits with a second seed on the **same calibration data** and matches the spatial patterns of the two decompositions (absolute cosine similarity, optimal one-to-one matching).
- `eog_relation` is the **relation-with-EOG** block: the scores of each excluded component with each EOG channel. It is independent of the seed comparison.
- `signal_quality` is the **reconstructed-signal** block: retained RMS amplitude and EEG–EOG correlation before/after cleaning, separately for the calibration runs (in-fit) and for the task runs (out-of-fit, unlabeled).
- `expected_fingerprint` summarises everything that determines the outputs (pipeline version, parameters, MNE version, hashes of the input files, ICA source). `outputs_compatible` reuses existing outputs **only** if the stored fingerprint equals the current one.

In [4]:
def fit_ica(raw_fit, seed):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        ica = ICA(n_components=N_ICA_COMPONENTS, max_iter='auto', method='infomax',
                  random_state=seed, fit_params=dict(extended=True))
        ica.fit(raw_fit, verbose=False)                                          # decomposes the 22 EEG channels
        idx, scores = ica.find_bads_eog(raw_fit, ch_name=EOG_NAMES, verbose=False)   # EOG used for scoring only
    ica.exclude = [int(i) for i in idx]
    return ica, np.asarray(scores), sorted({str(x.message)[:120] for x in w})

def ica_stability(ica_a, ica_b):
    '''BLOCK 2 - stability between seeds (same data, different seeds). Says nothing about the relation with the EOG.'''
    A, B = ica_a.get_components(), ica_b.get_components()                       # (22 channels, n_components)
    A, B = A / np.linalg.norm(A, axis=0), B / np.linalg.norm(B, axis=0)
    C = np.abs(A.T @ B)
    rows, cols = linear_sum_assignment(-C)
    match = dict(zip(rows.tolist(), cols.tolist()))
    excl = [dict(component=int(a), matched_component=int(match[a]), abs_cosine=round(float(C[a, match[a]]), 3),
                 matched_also_excluded=bool(match[a] in set(ica_b.exclude))) for a in ica_a.exclude]
    return dict(median_abs_cosine=round(float(np.median(C[rows, cols])), 3), min_abs_cosine=round(float(C[rows, cols].min()), 3),
                excluded_components=excl)

def eog_relation(ica, scores):
    '''BLOCK 3 - relation with the EOG channels: score of each component with each EOG channel (MNE find_bads_eog).'''
    S = np.abs(np.asarray(scores))                                              # (n_eog, n_components)
    others = [c for c in range(S.shape[1]) if c not in set(ica.exclude)]
    return dict(measure='absolute scores returned by mne find_bads_eog (default settings), one value per EOG channel',
                excluded_components=[dict(component=int(c), abs_score_per_eog={n: round(float(S[k, c]), 3) for k, n in enumerate(EOG_NAMES)},
                                          max_abs_score=round(float(S[:, c].max()), 3)) for c in ica.exclude],
                max_abs_score_non_excluded=round(float(S[:, others].max()), 3) if others else None)

def signal_quality(pre_runs, post_runs):
    '''BLOCK 4 - quality of the reconstructed signal, before vs after cleaning, on a set of runs.'''
    pre = np.concatenate([r.get_data() for r in pre_runs], axis=1)
    post = np.concatenate([r.get_data() for r in post_runs], axis=1)
    ratio = np.sqrt((post[:22] ** 2).mean(axis=1)) / np.sqrt((pre[:22] ** 2).mean(axis=1))      # retained RMS per EEG channel
    def mean_abs_corr(x):
        return float(np.abs(np.corrcoef(np.vstack([x[:22], x[22:]]))[:22, 22:]).mean())          # EEG x EOG
    return dict(n_samples=int(pre.shape[1]),
                rms_retained_ratio=dict(median=round(float(np.median(ratio)), 3), min=round(float(ratio.min()), 3),
                                        max=round(float(ratio.max()), 3)),
                mean_abs_corr_eeg_eog_before=round(mean_abs_corr(pre), 4), mean_abs_corr_eeg_eog_after=round(mean_abs_corr(post), 4))

def qc_flags(qc, non_standard):
    '''Descriptive flags, one list per block. Thresholds are conventions, not tuned; nothing is selected from them.'''
    f = dict(structure=[], seed_stability=[], eog_relation=[], reconstructed_signal=[])
    if qc['structure']['calibration_total_s'] < 60: f['structure'].append('short_calibration(<60s)')
    if non_standard: f['structure'].append('non_standard_calibration_structure')
    if qc['fit_counter']['fit_warnings']: f['structure'].append('ica_fit_warnings')
    if any((not e['matched_also_excluded']) or e['abs_cosine'] < 0.9 for e in qc['seed_stability']['excluded_components']):
        f['seed_stability'].append('excluded_component_not_reproduced_with_second_seed')
    n_ex = len(qc['eog_relation']['excluded_components'])
    if n_ex == 0: f['eog_relation'].append('no_ocular_component_excluded')
    if n_ex >= 5: f['eog_relation'].append('many_components_excluded(>=5 of 20)')
    if any(e['max_abs_score'] < 0.3 for e in qc['eog_relation']['excluded_components']):
        f['eog_relation'].append('excluded_component_with_weak_abs_eog_score(<0.3)')   # MNE's selection is relative (z-score)
    t = qc['reconstructed_signal']['task_runs']
    if t['mean_abs_corr_eeg_eog_after'] >= t['mean_abs_corr_eeg_eog_before']:
        f['reconstructed_signal'].append('eeg_eog_correlation_not_reduced_on_task_runs')
    if t['rms_retained_ratio']['min'] < 0.5:
        f['reconstructed_signal'].append('a_channel_lost_more_than_half_of_its_rms_on_task_runs')
    return f

def expected_fingerprint(sid, session):
    name = f'A{sid:02d}{session}'
    lab = LABELS_PATH / f'{name}.mat'
    fp = dict(pipeline_version=PIPELINE_VERSION, band_hz=list(BAND), filter_phase=FILTER_PHASE,
              n_ica_components=N_ICA_COMPONENTS, ica_seed=ICA_SEED, ica_stability_seed=ICA_STABILITY_SEED,
              ica_source_E_setting=ICA_SOURCE_E, overflow_gap=GAP, cue_after_start=CUE_AFTER_START, mne_version=mne.__version__,
              gdf_sha256=sha256_file(DATA_PATH / f'{name}.gdf'),
              labels_sha256=sha256_file(lab) if lab.exists() else None)
    if session == 'E' and ICA_SOURCE_E == 'session_T_calibration':
        t_ica = OUTPUT_PATH / f'A{sid:02d}T_calibration-ica.fif'
        fp['frozen_T_ica_sha256'] = sha256_file(t_ica) if t_ica.exists() else None
    return fp

def outputs_compatible(sid, session):
    '''True only if all outputs exist and their stored fingerprint equals the one of the current configuration.'''
    name = f'A{sid:02d}{session}'
    needed = [OUTPUT_PATH / f'{name}{CLEAN_SUFFIX}', OUTPUT_PATH / f'{name}_trials.csv', OUTPUT_PATH / f'{name}_provenance.json']
    if session == 'T' or ICA_SOURCE_E == 'own_calibration':
        needed.append(OUTPUT_PATH / f'{name}_calibration-ica.fif')
    missing = [p.name for p in needed if not p.exists()]
    if missing:
        return False, f'missing files {missing}'
    stored = json.loads((OUTPUT_PATH / f'{name}_provenance.json').read_text()).get('fingerprint', {})
    current = expected_fingerprint(sid, session)
    diff = [k for k in current if stored.get(k) != current[k]]
    return (not diff), ('compatible' if not diff else f'fingerprint differs in {diff}')

## Cleaning one session (run-wise filter, calibration-only ICA)

1. Each run → minimum-phase FIR 1–40 Hz (offline, MNE default padding).
2. ICA fitted **only on the calibration runs** (concatenated only for fitting), then frozen.
3. The frozen decomposition is applied to every run separately; the cleaned runs are concatenated.
4. Outputs: cleaned `.fif` (class annotations at the cue), trial table with run boundaries, provenance with fingerprint and ICA quality control.

In [5]:
def clean_session(sid, session):
    name = f'A{sid:02d}{session}'
    fingerprint = expected_fingerprint(sid, session)
    raw, pos, typ = read_gdf_session(DATA_PATH / f'{name}.gdf')
    data, starts, data_start, data_end = split_runs(raw, pos, typ)
    table, task_runs = build_trial_table(sid, session, pos, typ, data_start, data_end)
    t_start = np.sort(pos[typ == 768])
    calib_runs, calib_cond, non_standard = identify_calibration(sid, session, pos, typ, starts, data_start, data_end,
                                                                t_start, task_runs)

    # --- structural checks on the experimental design ---
    assert len(task_runs) == 6 and (table.groupby('run').size() == 48).all(), 'expected 6 task runs of 48 trials'
    if session == 'T':
        assert (table.groupby(['run', 'label_code']).size() == 12).all(), 'expected 12 trials per class per run'
    if non_standard:
        print(f'  note: {name} has a non-standard calibration structure {list(calib_cond.values())} (documented for A04T).')

    # --- 1) filtering run by run: no gap samples and no mixing between runs ---
    run_raws = []
    for r in range(len(data_start)):
        info = mne.create_info(CH_NAMES, FS, ['eeg'] * 22 + ['eog'] * 3)
        run_raws.append(mne.io.RawArray(data[:, data_start[r]:data_end[r]], info, verbose=False))
    run_f = [r.copy().filter(l_freq=BAND[0], h_freq=BAND[1], phase=FILTER_PHASE, fir_design='firwin', verbose=False)
             for r in run_raws]

    # --- 2) ICA fitted ONLY on the calibration runs (never on task runs, never with labels) ---
    calib_dur = [round((data_end[r] - data_start[r]) / FS, 1) for r in calib_runs]
    own_fit = (session == 'T') or (ICA_SOURCE_E == 'own_calibration')
    if own_fit:
        calib_raw = mne.concatenate_raws([run_f[i].copy() for i in calib_runs])   # concatenation only used for fitting
        ica, scores, warn_msgs = fit_ica(calib_raw, ICA_SEED)
        ica_b, _, _ = fit_ica(calib_raw, ICA_STABILITY_SEED)                       # same data, second seed
        qc = dict(structure=dict(calibration_conditions=list(calib_cond.values()), calibration_durations_s=calib_dur,
                                 calibration_total_s=round(float(sum(calib_dur)), 1), n_fit_samples=int(ica.n_samples_)),
                  # descriptive counter only: it does not identify why the algorithm stopped nor certify the decomposition
                  fit_counter=dict(n_iter=int(ica.n_iter_), max_iter=int(ica.max_iter), mne_version=mne.__version__,
                                   fit_warnings=warn_msgs),
                  seed_stability=ica_stability(ica, ica_b), eog_relation=eog_relation(ica, scores))
        ica.save(OUTPUT_PATH / f'{name}_calibration-ica.fif', overwrite=True)
        ica_source = f'{name} calibration runs'
    else:
        # strictest option: reuse the frozen decomposition fitted on the calibration runs of session T
        t_ica_path, t_prov_path = OUTPUT_PATH / f'A{sid:02d}T_calibration-ica.fif', OUTPUT_PATH / f'A{sid:02d}T_provenance.json'
        if not t_ica_path.exists():
            raise FileNotFoundError(f'{t_ica_path} not found: process session T first (same ICA_SOURCE_E setting).')
        ica = read_ica(t_ica_path, verbose=False)
        t_qc = json.loads(t_prov_path.read_text())['ica']['qc']
        qc = {k: t_qc[k] for k in ('structure', 'fit_counter', 'seed_stability', 'eog_relation')}   # properties of the T fit being reused
        qc['note'] = 'blocks 1-3 describe the session-T fit that is reused; block 4 is computed on this session'
        ica_source = f'A{sid:02d}T calibration runs (frozen, from the other session)'
    assert list(ica.ch_names) == CH_NAMES[:22]

    # --- 3) apply the frozen ICA to every run separately, then concatenate ---
    cleaned = []
    for r in run_f:
        c = r.copy(); ica.apply(c, verbose=False); cleaned.append(c)
    # BLOCK 4 - reconstructed-signal quality: calibration runs (used for the fit) and task runs (not used, unlabeled)
    qc['reconstructed_signal'] = dict(
        calibration_runs=signal_quality([run_f[i] for i in calib_runs], [cleaned[i] for i in calib_runs]),
        task_runs=signal_quality([run_f[i] for i in task_runs], [cleaned[i] for i in task_runs]))
    qc['flags'] = qc_flags(qc, non_standard)
    raw_clean = mne.concatenate_raws(cleaned)
    assert raw_clean.n_times == sum(e - s for s, e in zip(data_start, data_end))
    assert np.isfinite(raw_clean.get_data()).all(), 'NaN/inf in the cleaned recording'
    raw_clean.set_annotations(raw_clean.annotations + mne.Annotations(      # class description at the cue
        onset=table['cue_sample'].values / FS, duration=0.0,
        description=[CLASS_DESC[int(c)] for c in table['label_code']]))
    save_path = OUTPUT_PATH / f'{name}{CLEAN_SUFFIX}'
    raw_clean.save(save_path, overwrite=True)
    table.to_csv(OUTPUT_PATH / f'{name}_trials.csv', index=False)

    edge_margin_s = float((table['cue_sample'] + int(REQUIRED_SPAN_S[0] * FS) - table['run_start_sample']).min() / FS)
    prov = dict(file=save_path.name, session=session, pipeline_version=PIPELINE_VERSION, fingerprint=fingerprint,
                label_source=table['label_source'].iloc[0], n_trials=len(table), data_source='GDF', mne_version=mne.__version__,
                overflow_samples_removed=GAP * (len(data_start) - 1), cue_minus_trial_start_samples=CUE_AFTER_START,
                min_distance_baseline_start_from_run_start_s=round(edge_margin_s, 2),
                calibration_events_verified=True,
                filter=dict(l_freq=BAND[0], h_freq=BAND[1], phase=FILTER_PHASE, applied='per run, offline, default padding'),
                ica=dict(source=ica_source, fit_runs=calib_runs if own_fit else 'from session T',
                         n_components=N_ICA_COMPONENTS, random_state=ICA_SEED,
                         excluded_components=list(map(int, ica.exclude)), applied_to='all runs, frozen',
                         uses_task_runs_for_fit=False, uses_labels=False, qc=qc))
    (OUTPUT_PATH / f'{name}_provenance.json').write_text(json.dumps(prov, indent=2))
    return prov

## Batch processing (both sessions)

Existing outputs are reused only if `outputs_compatible` confirms that their provenance fingerprint matches the current configuration; otherwise they are recomputed and the reason is printed.

In [6]:
log_rows = []
for sid in SUBJECTS:
    for session in SESSIONS:
        name = f'A{sid:02d}{session}'
        if not (DATA_PATH / f'{name}.gdf').exists():
            print(f'[missing] {name}: no GDF file in {DATA_PATH}')
            continue
        ok, reason = outputs_compatible(sid, session)
        if SKIP_EXISTING and ok:
            print(f'[skip] {name}: existing outputs are compatible with the current configuration')
            continue
        if (OUTPUT_PATH / f'{name}_provenance.json').exists() or (OUTPUT_PATH / f'{name}{CLEAN_SUFFIX}').exists():
            print(f'[recompute] {name}: existing outputs not reused ({reason})')
        print(f'--- {name} ---')
        prov = clean_session(sid, session)
        q = prov['ica']['qc']
        log_rows.append(dict(subject=sid, session=session, ica_source=prov['ica']['source'],
                             calibration=q['structure']['calibration_conditions'], calibration_s=q['structure']['calibration_total_s'],
                             excluded=prov['ica']['excluded_components']))
        print(f"{name}: ICA from {prov['ica']['source']} | excluded {prov['ica']['excluded_components']} | flags {q['flags']}")

if log_rows:
    display(pd.DataFrame(log_rows))

--- A01T ---


c:\Users\ACER\anaconda3\envs\env_master_2025\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A01T: ICA from A01T calibration runs | excluded [0, 3, 2] | flags {'structure': [], 'seed_stability': [], 'eog_relation': ['excluded_component_with_weak_abs_eog_score(<0.3)'], 'reconstructed_signal': []}
--- A01E ---


c:\Users\ACER\anaconda3\envs\env_master_2025\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A01E: ICA from A01E calibration runs | excluded [0, 3] | flags {'structure': [], 'seed_stability': [], 'eog_relation': [], 'reconstructed_signal': []}
--- A02T ---


...

c:\Users\ACER\anaconda3\envs\env_master_2025\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A09T: ICA from A09T calibration runs | excluded [2, 3, 0] | flags {'structure': [], 'seed_stability': ['excluded_component_not_reproduced_with_second_seed'], 'eog_relation': ['excluded_component_with_weak_abs_eog_score(<0.3)'], 'reconstructed_signal': []}
--- A09E ---


c:\Users\ACER\anaconda3\envs\env_master_2025\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


A09E: ICA from A09E calibration runs | excluded [0, 3] | flags {'structure': [], 'seed_stability': [], 'eog_relation': [], 'reconstructed_signal': []}


,subject,session,ica_source,calibration,calibration_s,excluded
0,1,T,A01T calibration runs,"[eyes open, eyes closed, eye movements]",365.3,"[0, 3, 2]"
1,1,E,A01E calibration runs,"[eyes open, eyes closed, eye movements]",423.2,"[0, 3]"
2,2,T,A02T calibration runs,"[eyes open, eyes closed, eye movements]",383.9,"[1, 4, 2]"
3,2,E,A02E calibration runs,"[eyes open, eyes closed, eye movements]",325.9,"[0, 4, 3]"
4,3,T,A03T calibration runs,"[eyes open, eyes closed, eye movements]",317.3,"[2, 7]"
5,3,E,A03E calibration runs,"[eyes open, eyes closed, eye movements]",270.2,"[1, 7]"
6,4,T,A04T calibration runs,[eye movements],79.6,"[1, 0, 3]"
7,4,E,A04E calibration runs,"[eyes open, eyes closed, eye movements]",315.3,"[1, 0]"
8,5,T,A05T calibration runs,"[eyes open, eyes closed, eye movements]",419.7,"[1, 2, 3]"
9,5,E,A05E calibration runs,"[eyes open, eyes closed, eye movements]",394.6,"[0, 5, 2]"


## ICA quality-control report

Four **separate** tables, one per block, plus the descriptive flags. The ICA fit counter is shown only as a record together with the MNE version in use; it does not tell why the algorithm stopped and does not certify the decomposition. Nothing here is used to tune parameters, to choose seeds, or to choose settings from session-`E` metrics. Sessions with flags (for example `A04T`, whose calibration block is shorter) should be inspected by looking at the components and the cleaned signal.

In [7]:
qc_all = []
for sid in SUBJECTS:
    for session in SESSIONS:
        f = OUTPUT_PATH / f'A{sid:02d}{session}_provenance.json'
        if f.exists():
            qc_all.append((sid, session, json.loads(f.read_text())))
print('MNE version recorded in the outputs:', sorted({p['mne_version'] for _, _, p in qc_all}), '| in use now:', mne.__version__)

def _q(p): return p['ica']['qc']
t1 = pd.DataFrame([dict(subject=s, session=ss, ica_source=p['ica']['source'], calibration=_q(p)['structure']['calibration_conditions'],
                        calibration_total_s=_q(p)['structure']['calibration_total_s'], n_fit_samples=_q(p)['structure']['n_fit_samples'],
                        excluded=p['ica']['excluded_components'],
                        n_iter_record=f"{_q(p)['fit_counter']['n_iter']}/{_q(p)['fit_counter']['max_iter']}",
                        flags=_q(p)['flags']['structure']) for s, ss, p in qc_all])
t2 = pd.DataFrame([dict(subject=s, session=ss, median_abs_cosine=_q(p)['seed_stability']['median_abs_cosine'],
                        excluded_vs_second_seed=[(e['abs_cosine'], e['matched_also_excluded']) for e in _q(p)['seed_stability']['excluded_components']],
                        flags=_q(p)['flags']['seed_stability']) for s, ss, p in qc_all])
t3 = pd.DataFrame([dict(subject=s, session=ss,
                        abs_score_excluded=[(e['component'], e['abs_score_per_eog']) for e in _q(p)['eog_relation']['excluded_components']],
                        max_abs_score_non_excluded=_q(p)['eog_relation']['max_abs_score_non_excluded'],
                        flags=_q(p)['flags']['eog_relation']) for s, ss, p in qc_all])
def _rs(p, k): return _q(p)['reconstructed_signal'][k]
t4 = pd.DataFrame([dict(subject=s, session=ss,
                        calib_rms_min_median=(_rs(p, 'calibration_runs')['rms_retained_ratio']['min'], _rs(p, 'calibration_runs')['rms_retained_ratio']['median']),
                        calib_eog_corr_before_after=(_rs(p, 'calibration_runs')['mean_abs_corr_eeg_eog_before'], _rs(p, 'calibration_runs')['mean_abs_corr_eeg_eog_after']),
                        task_rms_min_median=(_rs(p, 'task_runs')['rms_retained_ratio']['min'], _rs(p, 'task_runs')['rms_retained_ratio']['median']),
                        task_eog_corr_before_after=(_rs(p, 'task_runs')['mean_abs_corr_eeg_eog_before'], _rs(p, 'task_runs')['mean_abs_corr_eeg_eog_after']),
                        flags=_q(p)['flags']['reconstructed_signal']) for s, ss, p in qc_all])
for title, t, fn in [('1. Structure and fit data', t1, 'ica_qc_1_structure.csv'), ('2. Stability between seeds', t2, 'ica_qc_2_seed_stability.csv'),
                     ('3. Relation with the EOG channels', t3, 'ica_qc_3_eog_relation.csv'), ('4. Reconstructed-signal quality', t4, 'ica_qc_4_signal_quality.csv')]:
    print(chr(10) + title); display(t); t.to_csv(OUTPUT_PATH / fn, index=False)

MNE version recorded in the outputs: ['1.12.0'] | in use now: 1.12.0

1. Structure and fit data


,subject,session,ica_source,calibration,calibration_total_s,n_fit_samples,excluded,n_iter_record,flags
0,1,T,A01T calibration runs,"[eyes open, eyes closed, eye movements]",365.3,91318,"[0, 3, 2]",500/500,[]
1,1,E,A01E calibration runs,"[eyes open, eyes closed, eye movements]",423.2,105790,"[0, 3]",500/500,[]
2,2,T,A02T calibration runs,"[eyes open, eyes closed, eye movements]",383.9,95959,"[1, 4, 2]",500/500,[]
3,2,E,A02E calibration runs,"[eyes open, eyes closed, eye movements]",325.9,81456,"[0, 4, 3]",500/500,[]
4,3,T,A03T calibration runs,"[eyes open, eyes closed, eye movements]",317.3,79320,"[2, 7]",500/500,[]
5,3,E,A03E calibration runs,"[eyes open, eyes closed, eye movements]",270.2,67565,"[1, 7]",500/500,[]
6,4,T,A04T calibration runs,[eye movements],79.6,19905,"[1, 0, 3]",500/500,[non_standard_calibration_structure]
7,4,E,A04E calibration runs,"[eyes open, eyes closed, eye movements]",315.3,78837,"[1, 0]",500/500,[]
8,5,T,A05T calibration runs,"[eyes open, eyes closed, eye movements]",419.7,104910,"[1, 2, 3]",121/500,[]
9,5,E,A05E calibration runs,"[eyes open, eyes closed, eye movements]",394.6,98653,"[0, 5, 2]",141/500,[]



2. Stability between seeds


,subject,session,median_abs_cosine,excluded_vs_second_seed,flags
0,1,T,0.999,"[(1.0, True), (1.0, True), (1.0, True)]",[]
1,1,E,0.981,"[(1.0, True), (1.0, True)]",[]
2,2,T,0.991,"[(1.0, True), (1.0, True), (1.0, True)]",[]
3,2,E,1.000,"[(1.0, True), (1.0, True), (1.0, True)]",[]
4,3,T,0.991,"[(1.0, True), (1.0, True)]",[]
5,3,E,0.997,"[(1.0, True), (1.0, True)]",[]
6,4,T,0.957,"[(1.0, True), (0.988, True), (0.961, False)]",[excluded_component_not_reproduced_with_second...
7,4,E,0.992,"[(1.0, True), (1.0, True)]",[]
8,5,T,0.949,"[(0.946, False), (1.0, True), (0.782, False)]",[excluded_component_not_reproduced_with_second...
9,5,E,0.991,"[(1.0, True), (1.0, True), (1.0, True)]",[]



3. Relation with the EOG channels


,subject,session,abs_score_excluded,max_abs_score_non_excluded,flags
0,1,T,"[(0, {'EOG-left': 0.091, 'EOG-central': 0.733,...",0.068,[excluded_component_with_weak_abs_eog_score(<0...
1,1,E,"[(0, {'EOG-left': 0.108, 'EOG-central': 0.808,...",0.113,[]
2,2,T,"[(1, {'EOG-left': 0.158, 'EOG-central': 0.894,...",0.078,[excluded_component_with_weak_abs_eog_score(<0...
3,2,E,"[(0, {'EOG-left': 0.038, 'EOG-central': 0.901,...",0.081,[excluded_component_with_weak_abs_eog_score(<0...
4,3,T,"[(2, {'EOG-left': 0.052, 'EOG-central': 0.695,...",0.204,[]
5,3,E,"[(1, {'EOG-left': 0.097, 'EOG-central': 0.786,...",0.107,[]
6,4,T,"[(1, {'EOG-left': 0.801, 'EOG-central': 0.111,...",0.134,[]
7,4,E,"[(1, {'EOG-left': 0.78, 'EOG-central': 0.126, ...",0.085,[]
8,5,T,"[(1, {'EOG-left': 0.199, 'EOG-central': 0.712,...",0.136,[]
9,5,E,"[(0, {'EOG-left': 0.09, 'EOG-central': 0.742, ...",0.179,[]



4. Reconstructed-signal quality


,subject,session,calib_rms_min_median,calib_eog_corr_before_after,task_rms_min_median,task_eog_corr_before_after,flags
0,1,T,"(0.363, 0.68)","(0.1429, 0.0174)","(0.514, 0.798)","(0.1492, 0.0224)",[]
1,1,E,"(0.496, 0.764)","(0.1293, 0.0259)","(0.752, 0.943)","(0.1292, 0.0239)",[]
2,2,T,"(0.547, 0.844)","(0.1045, 0.0329)","(0.554, 0.926)","(0.2281, 0.0377)",[]
3,2,E,"(0.42, 0.79)","(0.1287, 0.0164)","(0.683, 0.899)","(0.0718, 0.076)",[eeg_eog_correlation_not_reduced_on_task_runs]
4,3,T,"(0.696, 0.915)","(0.1512, 0.0383)","(0.579, 0.886)","(0.0658, 0.0846)",[eeg_eog_correlation_not_reduced_on_task_runs]
5,3,E,"(0.649, 0.913)","(0.1288, 0.0644)","(0.86, 0.981)","(0.0475, 0.0242)",[]
6,4,T,"(0.427, 0.637)","(0.1811, 0.0297)","(0.786, 0.95)","(0.1395, 0.0247)",[]
7,4,E,"(0.409, 0.667)","(0.1443, 0.0321)","(0.474, 0.767)","(0.1675, 0.0494)",[a_channel_lost_more_than_half_of_its_rms_on_t...
8,5,T,"(0.468, 0.735)","(0.2053, 0.0647)","(0.686, 0.926)","(0.2123, 0.0808)",[]
9,5,E,"(0.516, 0.766)","(0.1912, 0.0396)","(0.782, 0.96)","(0.1267, 0.0483)",[]


## Post-checks on the saved files

For every session the notebook **asserts**: channel names, order and types (22 EEG + 3 EOG); finite values; cue annotations equal to the trial table in **sample positions and in class descriptions**; label codes valid; run structure (6 runs of 48 trials); data around every cue inside the correct run; provenance fingerprint matching the current configuration.

In [8]:
def validate_outputs(sid, session, directory=None, suffix=None):
    directory = Path(directory) if directory is not None else OUTPUT_PATH
    name = f'A{sid:02d}{session}'
    raw = mne.io.read_raw_fif(directory / f'{name}{suffix or CLEAN_SUFFIX}', preload=True, verbose=False)
    table = pd.read_csv(directory / f'{name}_trials.csv')
    prov = json.loads((directory / f'{name}_provenance.json').read_text())

    # channels and types
    assert raw.ch_names == CH_NAMES, 'channel names/order differ from the official order'
    assert raw.get_channel_types() == ['eeg'] * 22 + ['eog'] * 3, 'channel types are not 22 EEG + 3 EOG'
    assert int(raw.info['sfreq']) == int(FS)
    assert np.isfinite(raw.get_data()).all(), 'NaN/inf in the cleaned recording'
    # cue annotations: positions AND classes must equal the trial table
    ann = raw.annotations
    keep = np.isin(ann.description, list(CLASS_DESC.values()))
    ann_samples = np.round(ann.onset[keep] * FS).astype(int)
    ann_desc = np.asarray(ann.description)[keep]
    order = np.argsort(ann_samples)
    tab = table.sort_values('cue_sample')
    assert len(ann_samples) == len(tab) == 288, 'wrong number of class annotations'
    assert np.array_equal(ann_samples[order], tab['cue_sample'].values), 'annotation positions differ from the trial table'
    assert list(ann_desc[order]) == [CLASS_DESC[int(c)] for c in tab['label_code']], 'annotation classes differ from the trial table'
    assert table['label_code'].isin([1, 2, 3, 4]).all()
    # run structure and containment
    assert sorted(table['run'].unique()) == list(range(6)) and (table.groupby('run').size() == 48).all()
    assert (table['cue_sample'] - table['trial_start_sample'] == CUE_AFTER_START).all()
    assert (table['cue_sample'] + int(REQUIRED_SPAN_S[0] * FS) >= table['run_start_sample']).all()
    assert (table['cue_sample'] + int(REQUIRED_SPAN_S[1] * FS) < table['run_end_sample']).all()
    assert table['run_end_sample'].max() <= raw.n_times
    # provenance
    assert prov['pipeline_version'] == PIPELINE_VERSION and prov['calibration_events_verified'] is True
    assert prov['ica']['uses_task_runs_for_fit'] is False and prov['ica']['uses_labels'] is False
    return dict(file=f'{name}', ch_ok=True, types_ok=True, finite=True, cue_positions_and_classes_match=True,
                class_counts=table['label_code'].value_counts().sort_index().tolist(),
                artifact_trials=int(table['artifact_flag'].sum()),
                min_baseline_distance_from_run_start_s=prov['min_distance_baseline_start_from_run_start_s'],
                ica_fit_runs=prov['ica']['fit_runs'])

rows = []
for sid in SUBJECTS:
    for session in SESSIONS:
        if (OUTPUT_PATH / f'A{sid:02d}{session}{CLEAN_SUFFIX}').exists():
            ok, reason = outputs_compatible(sid, session)
            assert ok, f'A{sid:02d}{session}: outputs not compatible with the current configuration ({reason})'
            rows.append(validate_outputs(sid, session))
display(pd.DataFrame(rows))

,file,ch_ok,types_ok,finite,cue_positions_and_classes_match,class_counts,artifact_trials,min_baseline_distance_from_run_start_s,ica_fit_runs
0,A01T,True,True,True,True,"[72, 72, 72, 72]",15,2.0,"[0, 1, 2]"
1,A01E,True,True,True,True,"[72, 72, 72, 72]",7,2.0,"[0, 1, 2]"
2,A02T,True,True,True,True,"[72, 72, 72, 72]",18,2.0,"[0, 1, 2]"
3,A02E,True,True,True,True,"[72, 72, 72, 72]",5,2.0,"[0, 1, 2]"
4,A03T,True,True,True,True,"[72, 72, 72, 72]",18,2.0,"[0, 1, 2]"
5,A03E,True,True,True,True,"[72, 72, 72, 72]",15,2.0,"[0, 1, 2]"
6,A04T,True,True,True,True,"[72, 72, 72, 72]",26,2.0,[0]
7,A04E,True,True,True,True,"[72, 72, 72, 72]",60,2.0,"[0, 1, 2]"
8,A05T,True,True,True,True,"[72, 72, 72, 72]",26,2.0,"[0, 1, 2]"
9,A05E,True,True,True,True,"[72, 72, 72, 72]",12,2.0,"[0, 1, 2]"
